# ARC-AGI-3 Duck Harness: Qwen -> Gemma dual-model mid-game switch

Every execution runs the real Duck Harness end to end. A normal Kaggle
run uses all 25 mounted local `environment_files`; an official competition
rerun discovers and executes every game exposed by the official gateway.
Every game runs exactly once: Qwen acts from start to finish, and when
Qwen stalls (no score/level progress within a policy budget) the shared
engine swaps to Gemma mid-run, Gemma reviews that same game's same-run
transcript as plain text, the engine swaps back to Qwen, and Qwen
continues the same environment pass seeded with the handoff.

In [1]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# In submission, disable the periodic JSON/HTML diagnostics writes and per-frame logging.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")


## 2. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).

In [2]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)


## 3. Locate the source bundle

Find the uploaded TAAF source dataset by its marker file, and record where Kaggle mounted
every attached input so setup commands and the solver can find them.

In [3]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["jeroencottaar/taaf-kaggle-source-share", "driessmit1/arc3-vllm-h100-wheelhouse-v3", "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir() -> Path:
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        return marker.parent
    raise RuntimeError("TAAF source bundle not found under /kaggle/input.")


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir()
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")


## 4. Import the bundled source and run solver setup

Put the snapshotted repositories on the path (this process and any child processes), then run
the solver's setup commands — installing wheels, fetching model weights, and so on.

In [4]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# Solver setup commands (wheels, vLLM server startup, ...) run before the benchmark loads.
env = _command_env()
for command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    print(f"taaf.kaggle: setup command: {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    # Re-read in case the command persisted new env keys.
    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# Bound generation before inference modules are imported. The original run
# allowed unlimited tool steps/output, amplifying stalls under high fan-out.
_score_runtime_env = {
    'LOCAL_ANALYZER_MAX_OUTPUT': os.environ.get('TAAF_MAX_OUTPUT_TOKENS', '8192'),
    'LOCAL_ANALYZER_TOOL_STEPS': os.environ.get('TAAF_TOOL_STEPS', '8'),
    'LOCAL_ANALYZER_TEMPERATURE': os.environ.get('TAAF_TEMPERATURE', '0.6'),
    'LOCAL_ANALYZER_TOP_P': os.environ.get('TAAF_TOP_P', '0.95'),
}
os.environ.update(_score_runtime_env)
_persisted_setup_env = json.loads(SETUP_ENV_PATH.read_text())
_persisted_setup_env.update(_score_runtime_env)
SETUP_ENV_PATH.write_text(json.dumps(_persisted_setup_env, indent=2, sort_keys=True) + '\n')
print(f'taaf.kaggle: bounded analyzer controls = {_score_runtime_env}')


## 4.1 Minimal Duck Harness compatibility

Keep the bundled solver unchanged except for the missing neutral `ACTION7` reverse mapping. No prompt, score, environment, or execution method is patched.


In [5]:
import inference.agent.action_names as action_names
import inference.framework.solver as solver_module

action_names.MODEL_TO_ENGINE_ACTION["ACTION7"] = "ACTION7"
assert action_names.to_model_action("ACTION7") == "ACTION7"
assert action_names.to_engine_action("ACTION7") == "ACTION7"

PATCH_STATUS = {
    "patch": "minimal-action7-reverse-map-v1",
    "action7_reverse_mapping": True,
    "system_prompt_changed": False,
    "solver_methods_changed": False,
    "dataset_modified": False,
}
print(f"taaf.kaggle: compatibility={PATCH_STATUS}")


## 4.2 Hard no-prior runtime contract

Declare and verify the information boundary before loading the benchmark. Only current-game observations, actions, rewards, transitions, Qwen's pass-1 transcript, and Gemma's review of that transcript may cross into pass 2.


In [6]:
NO_PRIOR_CONTRACT = {
    "allowed": [
        "same_current_run_same_game_observations",
        "same_current_run_same_game_actions",
        "same_current_run_same_game_rewards",
        "same_current_run_same_game_transitions",
        "qwen_pass1_transcript_for_same_game",
        "gemma_review_of_that_same_transcript",
    ],
    "forbidden": [
        "historical_transcripts",
        "yesterday_transcripts",
        "routebooks",
        "replays",
        "solved_paths",
        "hidden_labels",
        "cross_run_state",
        "cross_game_state",
    ],
}
assert set(NO_PRIOR_CONTRACT["allowed"]).isdisjoint(
    NO_PRIOR_CONTRACT["forbidden"]
)
print("taaf.kaggle: strict no-prior contract active")


## 5. Load the benchmark

Unpickle the deployment target and the benchmark, stamping the real submission state onto the
target and pointing the benchmark's outputs at the Kaggle working directory.

In [7]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR


## 6. Fixed real-run configuration

Run every discovered game with concurrency 4 and strict no-prior behavior. A dynamic per-game cap keeps both complete passes inside Kaggle's nine-hour GPU runtime limit. No environment can be omitted.


In [8]:
STRICT_NO_PRIOR = True
TARGET_CONCURRENCY = 4

_original_game_budget = float(
    getattr(bm.solver, "max_runtime_s_per_game", 0.0) or 0.0
)
_original_concurrency = max(
    1, int(getattr(bm.solver, "concurrency", 1) or 1)
)
bm.solver.concurrency = TARGET_CONCURRENCY

# Optional grafts are constrained to the same current game and current run.
# Banking and transfer are intentionally never installed.
try:
    from taaf_grafts.composite import install as _install_taaf_grafts
except ModuleNotFoundError:
    _install_taaf_grafts = None

_graft_flags = {
    "shortcircuit": True,
    "efficiency": True,
    "retry_guard": True,
    "recovery": True,
    "context_window": int(
        os.environ.get("TAAF_CONTEXT_WINDOW", "32768")
    ),
}
if _install_taaf_grafts is not None:
    _install_taaf_grafts(bm, _graft_flags, expected_version=1)

assert bm.solver.concurrency == TARGET_CONCURRENCY
assert "banking" not in _graft_flags
assert "transfer" not in _graft_flags
print(
    "REAL RUN CONFIG: "
    f"strict_no_prior={STRICT_NO_PRIOR} "
    f"concurrency={TARGET_CONCURRENCY} "
    "games_required=all_discovered "
    f"source_per_game_budget={_original_game_budget}"
)


## 6.2 Before the switch run

All environment passes below are single passes. The relay in the base
notebook ran every game twice (Qwen pass 1, Gemma review, Qwen pass 2)
and kept the better pass; this notebook instead keeps one pass per game
and swaps to Gemma for a same-run review when the primary stalls, then
returns to Qwen, which doubles the per-game time budget at the same
concurrency.

## 6.1 Dual-model mid-game switch module

`DualModelSwitchToolAgent` starts every game on Qwen and watches the
analyzer's own action results. When Qwen makes no score/level progress
within a policy budget, the shared engine swaps to Gemma mid-run, Gemma
reviews the same-run same-game transcript (plain text, no tools), the
engine swaps back to Qwen, and Qwen continues the SAME environment pass
seeded with that handoff. No prior or cross-game data is used.

In [ ]:
"""Dual-model mid-game switch for the ARC-AGI-3 Duck Harness (TAAF).

Single environment pass per game. The primary model (Qwen) acts in every game
from start to finish. A stall policy watches the analyzer's own action results;
when Qwen stops making score/level progress for a policy budget, the shared
engine is temporarily swapped to the secondary model (Gemma), Gemma reviews
the SAME run's, SAME game's transcript as a plain text handoff (no tools, no
images -- the proven Duck Harness relay pattern), and the engine swaps back to
Qwen, which continues the same environment pass seeded with that handoff.
No cross-game or historical data is ever used.

The switch logic is engine-agnostic: with `MidgameEngineController` wired to a
notebook callback, the controller performs the actual server swap (stop Qwen /
start Gemma, then back) under a lock so every concurrent analyzer either waits
or continues once the engine returns to Qwen before its next LLM call. In "hot"
mode both servers are already up and the callbacks are no-ops.

Pure Python: imports only the TAAF `inference.agent.tool_agent` package, which
is on `sys.path` inside the Duck Harness notebook (cell 7) and in the local
test harness.
"""
from __future__ import annotations

import json
import logging
import threading
import time
from pathlib import Path
from typing import Any, Callable, Optional

from inference.agent.tool_agent import AnalyzerModelConfig, ToolAgent

log = logging.getLogger("duck_midgame")


class _RWLock:
    """Reader-writer lock with writer preference.

    Acting completions take the read side so concurrent games share the Qwen
    engine (like the proven relay). Engine swaps take the write side so no
    request is in flight while the server is being replaced.
    """

    def __init__(self) -> None:
        self._cond = threading.Condition()
        self._readers = 0
        self._writer = False
        self._write_waiters = 0

    def read(self):
        def acquire():
            with self._cond:
                while self._writer or self._write_waiters:
                    self._cond.wait()
                self._readers += 1

        def release():
            with self._cond:
                self._readers -= 1
                if self._readers == 0:
                    self._cond.notify_all()

        return _LockHandle(acquire, release)

    def write(self):
        def acquire():
            with self._cond:
                self._write_waiters += 1
                try:
                    while self._writer or self._readers:
                        self._cond.wait()
                    self._writer = True
                finally:
                    self._write_waiters -= 1

        def release():
            with self._cond:
                self._writer = False
                self._cond.notify_all()

        return _LockHandle(acquire, release)


class _LockHandle:
    __slots__ = ("_acquire", "_release")

    def __init__(self, acquire, release) -> None:
        self._acquire = acquire
        self._release = release

    def __enter__(self):
        self._acquire()
        return self

    def __exit__(self, exc_type, exc, tb):
        self._release()
        return False


# --------------------------------------------------------------------------
# Engine controller: one shared engine for all concurrent games
# --------------------------------------------------------------------------


class MidgameEngineController:
    """Tracks the shared serving engine and serializes switches.

    `active` is "primary" or "secondary". `to_secondary` / `to_primary` are
    notebook-provided callbacks that perform the actual server swaps (or no-ops
    in hot mode). All analyzers take `engine.lock` while syncing model configs;
    a switch callback runs under the same lock, so concurrent analyzers block on
    their next completion until the review cycle finishes and the engine returns
    to the primary model.
    """

    def __init__(
        self,
        *,
        primary: dict[str, str],
        secondary: dict[str, str],
        to_secondary: Callable[[], None],
        to_primary: Callable[[], None],
    ) -> None:
        if not callable(to_secondary) or not callable(to_primary):
            raise TypeError("EngineController requires real to_secondary/to_primary callbacks")
        self.primary = _config(primary, "primary")
        self.secondary = _config(secondary, "secondary")
        self.active = "primary"
        self.switch_count = 0
        self.lock = threading.RLock()  # internal switch bookkeeping
        self.rwlock = _RWLock()        # acting completions (read) vs swaps (write)
        self._to_secondary = to_secondary
        self._to_primary = to_primary

    def switch_to_secondary(self) -> None:
        # Caller holds the write side of rwlock for the whole swap cycle.
        if self.active == "secondary":
            return
        self._to_secondary()
        self.active = "secondary"
        self.switch_count += 1
        print(
            f"MIDGAME SWITCH ENGINE primary -> secondary count={self.switch_count}",
            flush=True,
        )
        log.info("ENGINE SWITCH primary -> secondary (count=%d)", self.switch_count)

    def switch_to_primary(self) -> None:
        # Caller holds the write side of rwlock for the whole swap cycle.
        if self.active == "primary":
            return
        self._to_primary()
        self.active = "primary"
        self.switch_count += 1
        print(
            f"MIDGAME SWITCH ENGINE secondary -> primary count={self.switch_count}",
            flush=True,
        )
        log.info("ENGINE SWITCH secondary -> primary (count=%d)", self.switch_count)

    def config_for(self, name: str) -> AnalyzerModelConfig:
        return self.primary if name == "primary" else self.secondary


def _config(raw: Any, name: str) -> AnalyzerModelConfig:
    if isinstance(raw, AnalyzerModelConfig):
        return raw
    if not isinstance(raw, dict):
        raise TypeError(f"{name} model config must be a dict or AnalyzerModelConfig")
    missing = {"provider", "base_url", "model_id"} - set(raw)
    if missing:
        raise ValueError(f"{name} model config missing keys: {sorted(missing)}")
    return AnalyzerModelConfig(
        provider=str(raw["provider"]).strip(),
        base_url=str(raw["base_url"]).strip(),
        model_id=str(raw["model_id"]).strip(),
    )


# --------------------------------------------------------------------------
# Stall policy
# --------------------------------------------------------------------------

DEFAULT_SWITCH_POLICY = {
    # Minimum executed environment actions before a switch is even considered.
    "min_actions_at_all": 30,
    # Executed actions with no score/level progress that trigger a switch.
    "min_actions_before_switch": 90,
    # Consecutive analysis steps with no progress that trigger a switch.
    "min_steps_before_switch": 12,
    # Maximum review cycles per game (1 = Qwen -> Gemma review -> Qwen once).
    "max_switches": 1,
    # Same-run transcript tail (chars) handed to the secondary model.
    "handoff_chars": 60_000,
    # Assistant turns of raw message history kept after the switch.
    "keep_recent_turns": 2,
    # Tokens budgeted for the secondary model's review of the transcript.
    "review_max_tokens": 800,
    # Minimum remaining per-game budget (seconds) required to attempt a switch.
    "min_remaining_budget_seconds": 0,
    # Board changes discount this many stall steps (exploration activity).
    "board_change_step_discount": 2,
}


def _policy_get(policy: dict[str, Any], key: str, default: Any) -> Any:
    return policy.get(key, default)


# --------------------------------------------------------------------------
# Dual-model switch agent
# --------------------------------------------------------------------------


class DualModelSwitchToolAgent(ToolAgent):
    """ToolAgent that runs a Gemma review cycle mid-game on a stall.

    Qwen acts for the whole run. Each `analyze` step updates stall/progress
    counters from the analyzer's own last action result. On trigger, the shared
    engine swaps to Gemma, Gemma reviews this game's same-run transcript (plain
    text, no tools), the engine swaps back to Qwen, and Qwen continues the SAME
    environment run seeded with the handoff and trimmed history.
    """

    def __init__(
        self,
        *,
        primary: Any,
        secondary: Any,
        engine: Optional[MidgameEngineController] = None,
        policy: Optional[dict[str, Any]] = None,
        **kwargs: Any,
    ) -> None:
        self._primary_config = _config(primary, "primary")
        self._secondary_config = _config(secondary, "secondary")
        super().__init__(
            model=self._primary_config.model_id,
            base_url=self._primary_config.base_url,
            provider=self._primary_config.provider,
            **kwargs,
        )
        self._engine = engine
        self._policy = dict(DEFAULT_SWITCH_POLICY)
        if policy:
            self._policy.update(policy)

        # stall / switch state
        self._max_score: int = -1
        self._stall_actions = 0
        self._stall_steps = 0
        self._switched = False
        self._switch_count = 0
        self._switch_action_num: Optional[int] = None
        self._switch_analysis_step: Optional[int] = None
        self._switch_stall_actions = 0
        self._switch_stall_steps = 0
        self._handoff: Optional[str] = None
        self._handoff_chars = 0
        self._in_review = False
        self._game_key: Optional[str] = None

    # -- public diagnostics -------------------------------------------------

    @property
    def active_model_id(self) -> str:
        return self._model.model_id

    @property
    def is_switched(self) -> bool:
        return self._switched

    def switch_manifest(self) -> dict[str, Any]:
        return {
            "game_key": self._game_key,
            "switched": self._switched,
            "switch_count": self._switch_count,
            "switch_action_num": self._switch_action_num,
            "switch_analysis_step": self._switch_analysis_step,
            "switch_stall_actions": self._switch_stall_actions,
            "switch_stall_steps": self._switch_stall_steps,
            "max_score_seen": self._max_score,
            "stall_actions": self._stall_actions,
            "stall_steps": self._stall_steps,
            "handoff_chars": self._handoff_chars,
            "primary_model": self._primary_config.model_id,
            "secondary_model": self._secondary_config.model_id,
            "active_model": self._model.model_id,
            "engine_active": self._engine.active if self._engine is not None else "n/a",
        }

    # -- analyze hook -------------------------------------------------------

    def analyze(
        self,
        state_path: Path,
        action_num: int,
        valid_actions: Optional[list[str]] = None,
        step_env: Optional[Callable[[dict[str, Any]], dict[str, Any]]] = None,
        transcript_path: Optional[Path] = None,
        analysis_step: Optional[int] = None,
        transcript_updated: Optional[Callable[[str], None]] = None,
        request_timeout_seconds: Optional[float] = None,
        should_stop: Optional[Callable[[], bool]] = None,
    ) -> Any:
        self._game_key = _stem_key(transcript_path) if transcript_path is not None else self._game_key
        self._observe_progress()
        self._maybe_switch(
            action_num=action_num,
            analysis_step=analysis_step,
            transcript_path=transcript_path,
            request_timeout_seconds=request_timeout_seconds,
        )
        return super().analyze(
            state_path,
            action_num,
            valid_actions=valid_actions,
            step_env=step_env,
            transcript_path=transcript_path,
            analysis_step=analysis_step,
            transcript_updated=transcript_updated,
            request_timeout_seconds=request_timeout_seconds,
            should_stop=should_stop,
        )

    # -- stall/progress bookkeeping ----------------------------------------

    def _observe_progress(self) -> None:
        result = self._last_action_result
        summary = self._last_step_summary
        if not isinstance(result, dict) and not isinstance(summary, dict):
            return

        made_progress = False
        executed = 0
        board_changed = False

        if isinstance(summary, dict):
            if summary.get("level_transition") or summary.get("run_complete"):
                made_progress = True
            try:
                executed = max(0, int(summary.get("executed_count") or 0))
            except (TypeError, ValueError):
                executed = 0
            board_changed = bool(summary.get("board_changed"))

        if isinstance(result, dict) and result.get("executed"):
            try:
                score = int(result.get("score"))
            except (TypeError, ValueError):
                score = self._max_score
            if score > self._max_score:
                self._max_score = score
                made_progress = True
            if result.get("level_completed") or result.get("run_complete"):
                made_progress = True
            board_changed = board_changed or bool(result.get("board_changed"))
            if not executed:
                try:
                    executed = max(0, int(result.get("executed_count") or 0))
                except (TypeError, ValueError):
                    executed = 0

        if made_progress:
            self._stall_actions = 0
            self._stall_steps = 0
            return

        # No progress this step.
        if executed > 0:
            self._stall_actions += executed
        if board_changed:
            discount = int(
                _policy_get(self._policy, "board_change_step_discount", 2) or 0
            )
            self._stall_steps = max(0, self._stall_steps - discount)
        self._stall_steps += 1

    # -- switch decision ----------------------------------------------------

    def _maybe_switch(
        self,
        *,
        action_num: int,
        analysis_step: Optional[int],
        transcript_path: Optional[Path],
        request_timeout_seconds: Optional[float],
    ) -> None:
        if self._engine is None:
            return
        if self._engine.active != "primary":
            # Another game is mid-review-cycle; the engine lock serializes our
            # next completion after it returns to Qwen, so just keep acting.
            return
        if self._switch_count >= int(_policy_get(self._policy, "max_switches", 1)):
            return

        min_actions_at_all = int(_policy_get(self._policy, "min_actions_at_all", 30))
        if action_num < min_actions_at_all:
            return
        min_actions = int(_policy_get(self._policy, "min_actions_before_switch", 90))
        min_steps = int(_policy_get(self._policy, "min_steps_before_switch", 12))
        if self._stall_actions < min_actions and self._stall_steps < min_steps:
            return
        # A mid-run engine swap blocks every concurrent lane for the swap +
        # review cycle (about 10 minutes on this stack). Only switch when the
        # remaining budget can still use the handoff; request_timeout_seconds
        # from the harness is min(analyzer timeout, remaining per-game budget).
        min_remaining = int(_policy_get(self._policy, "min_remaining_budget_seconds", 0) or 0)
        if min_remaining > 0 and request_timeout_seconds is not None:
            if request_timeout_seconds < min_remaining:
                log.info(
                    "MIDGAME SWITCH skip: game=%s remaining=%.0fs < %ds",
                    self._game_key, request_timeout_seconds, min_remaining,
                )
                return

        # Persist the stall snapshot for diagnostics before the swap.
        self._switch_action_num = action_num
        self._switch_analysis_step = analysis_step
        self._switch_stall_actions = self._stall_actions
        self._switch_stall_steps = self._stall_steps

        print(
            "MIDGAME SWITCH TRIGGER "
            f"game={self._game_key} action={action_num} step={analysis_step} "
            f"stall_actions={self._stall_actions} stall_steps={self._stall_steps}",
            flush=True,
        )
        log.info(
            "MIDGAME SWITCH TRIGGER game=%s action=%s step=%s stall_actions=%s stall_steps=%s",
            self._game_key,
            action_num,
            analysis_step,
            self._stall_actions,
            self._stall_steps,
        )
        with self._engine.rwlock.write():
            # Exclusive: no acting completion is in flight while the engine is
            # swapped to Gemma for this review, and concurrent games simply
            # wait until Qwen is back before their next completion.
            if self._engine.active == "primary":
                self._engine.switch_to_secondary()
            self._adopt_secondary(
                handoff=True,
                transcript_path=transcript_path,
                request_timeout_seconds=request_timeout_seconds,
            )
            # Review done: hand the engine back to Qwen so this game and every
            # concurrent game continue acting with the primary model.
            if self._engine.active != "primary":
                self._engine.switch_to_primary()

    # -- engine adoption ----------------------------------------------------

    def _sync_before_completion(self) -> None:
        """Align the acting model with the engine. Caller holds the read side."""
        if self._engine is None:
            return
        if self._in_review:
            # A review call: keep the secondary model for this one completion.
            return
        self._model = self._primary_config

    def _adopt_secondary(
        self,
        *,
        handoff: bool,
        transcript_path: Optional[Path] = None,
        request_timeout_seconds: Optional[float] = None,
    ) -> Optional[str]:
        first_adoption = not self._switched
        self._switched = True
        self._switch_count += 1
        review: Optional[str] = None
        if handoff:
            self._model = self._secondary_config
            self._in_review = True
            try:
                review = self._review_same_run_transcript(
                    transcript_path=transcript_path,
                    request_timeout_seconds=request_timeout_seconds,
                )
            finally:
                self._in_review = False
                self._model = self._primary_config
            if review:
                self._handoff = review
                self._handoff_chars = len(review)
                self._summarized_knowledge.update(
                    {
                        "world_model": review,
                        "recent_findings": (
                            "Mid-game model switch: Gemma reviewed this same "
                            "game's transcript because Qwen stalled; verify the "
                            "handoff against the live current_frame before "
                            "continuing."
                        ),
                        "current_plan": (
                            "Use the same-run handoff to avoid repeated failed "
                            "probes, then continue searching and acting through "
                            "the normal Duck Harness tools."
                        ),
                    }
                )
        if first_adoption:
            self._system_prompt += (
                "\n\nThis run continues after a mid-game Gemma review of this "
                "same game. Treat the supplied handoff as hypotheses, verify "
                "them against the fresh current_frame, and revise when live "
                "evidence disagrees. No prior-run, replay, routebook, "
                "other-game, or hidden-label information is available."
            )
            if int(_policy_get(self._policy, "keep_recent_turns", 2) or 0) > 0:
                keep = int(self._policy["keep_recent_turns"])
                self._history_messages = self._keep_recent_history_turns(
                    self._history_messages,
                    max_turns=keep,
                )
        self._stall_actions = 0
        self._stall_steps = 0
        return review

    # -- same-run handoff ---------------------------------------------------

    def _review_same_run_transcript(
        self,
        *,
        transcript_path: Optional[Path],
        request_timeout_seconds: Optional[float],
    ) -> Optional[str]:
        if transcript_path is None:
            return None
        path = Path(transcript_path)
        if not path.exists():
            return None
        tail_chars = int(_policy_get(self._policy, "handoff_chars", 60_000) or 0)
        transcript = path.read_text(encoding="utf-8", errors="replace")
        if tail_chars > 0:
            transcript = transcript[-tail_chars:]
        stem = _stem_key(path)
        review_max_tokens = int(_policy_get(self._policy, "review_max_tokens", 800) or 0)
        previous_max_output = self._max_output_tokens
        if review_max_tokens > 0:
            self._max_output_tokens = max(1, review_max_tokens)
        prompt = f"""
You are the reviewing model in a strict no-prior Duck Harness mid-game switch.
This is game {stem}. You may use ONLY the transcript below, produced moments
ago in this same competition run and same game, plus nothing else. Do not
import remembered routes, prior submissions, other games, hidden labels, or
external knowledge about this environment.

Qwen stalled on this game. Review its observed objects, tested actions,
rewards, state changes, failures, and partial world model. Return a compact
handoff for Qwen to continue this SAME environment run. Include:
- World model
- Goal model
- Action model
- Confirmed findings
- Failed hypotheses or moves to avoid
- Best next plan
Do not emit tool calls and do not claim facts absent from the transcript.

PASS-1 QWEN TRANSCRIPT FOR GAME {stem}:
{transcript}
""".strip()
        last_error: Optional[BaseException] = None
        content = ""
        for attempt in range(1, 3):
            try:
                result = self._chat_completion(
                    [{"role": "user", "content": prompt}],
                    tools=None,
                    request_timeout_seconds=request_timeout_seconds,
                )
            except Exception as exc:  # network/server errors
                last_error = exc
                log.warning(
                    "MIDGAME SWITCH review retry game=%s attempt=%d error=%r",
                    stem, attempt, exc,
                )
                time.sleep(5 * attempt)
                continue
            try:
                message = getattr(result, "message", None)
                if isinstance(message, dict):
                    content = str(message.get("content") or "").strip()
                else:
                    content = str(message or "").strip()
            except Exception as exc:  # pragma: no cover - malformed result
                last_error = exc
                content = ""
            if content:
                break
            log.warning(
                "MIDGAME SWITCH empty handoff game=%s attempt=%d; retrying",
                stem, attempt,
            )
            time.sleep(5 * attempt)
        self._max_output_tokens = previous_max_output
        if not content:
            log.warning(
                "MIDGAME SWITCH review failed for %s after 2 attempts: %r; continuing without it",
                stem, last_error,
            )
            return None
        print(
            "MIDGAME SWITCH handoff complete: "
            f"game={stem} chars={len(content)}",
            flush=True,
        )
        log.info(
            "MIDGAME SWITCH handoff complete: game=%s chars=%d",
            stem,
            len(content),
        )
        return content

    # -- completion hook ----------------------------------------------------

    def _chat_completion(
        self,
        messages: list[dict[str, Any]],
        *,
        tools: Optional[list[dict[str, Any]]] = None,
        request_timeout_seconds: Optional[float] = None,
    ) -> Any:
        # Acting completions share the read side so all concurrent games keep
        # the engine busy (the proven relay behavior). The Gemma review call
        # runs inside a swap cycle that already holds the write side, so it
        # bypasses the rwlock via _in_review.
        if self._engine is None:
            return super()._chat_completion(
                messages,
                tools=tools,
                request_timeout_seconds=request_timeout_seconds,
            )
        if self._in_review:
            return super()._chat_completion(
                messages,
                tools=tools,
                request_timeout_seconds=request_timeout_seconds,
            )
        with self._engine.rwlock.read():
            self._sync_before_completion()
            return super()._chat_completion(
                messages,
                tools=tools,
                request_timeout_seconds=request_timeout_seconds,
            )


def _stem_key(path: Optional[Path]) -> str:
    if path is None:
        return "unknown"
    name = Path(path).name
    return str(name).split("-", 1)[0]


# --------------------------------------------------------------------------
# Notebook helper: manifest writer
# --------------------------------------------------------------------------


def write_switch_manifest(agents: list[DualModelSwitchToolAgent], path: Path) -> None:
    payload = {
        "protocol": "duck_harness_dual_model_midgame_switch_v2",
        "environment_passes": 1,
        "per_game": [agent.switch_manifest() for agent in agents],
        "switched_games": [agent._game_key for agent in agents if agent._switched],
    }
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return None

# Register this cell's definitions as the `duck_midgame` module so the
# switch cell can `from duck_midgame import ...`.
import sys as _duck_sys
import types as _duck_types
_duck_midgame_module = _duck_types.ModuleType("duck_midgame")
_duck_midgame_module.__dict__.update(globals())
_duck_sys.modules["duck_midgame"] = _duck_midgame_module


## 7. Real single-pass dual-model mid-game switch

Normal execution uses the mounted local Arcade; competition execution
uses the official gateway. Either way every game runs exactly once:
Qwen acts every game; on a stall the engine swaps to Gemma for a
same-run text review, then back to Qwen, which continues the same pass.

In [ ]:
# Exactly two Duck Harness environment runs per game:
#   pass 1: Qwen observes and acts
#   relay:  Gemma reviews only that same-game, same-current-run transcript
#   pass 2: Qwen receives Gemma's review and observes/acts in a fresh run
# The final output keeps the higher-scoring Qwen run for each game.
import copy
import re
import shutil
import signal
from urllib.request import Request

FIRST_PASS_DIR = WORKING_DIR / "relay_pass1_qwen"
GEMMA_REVIEW_DIR = WORKING_DIR / "relay_gemma_reviews"
GEMMA_RUNTIME_DIR = WORKING_DIR / "gemma-llamacpp-bin"
FIRST_PASS_DIR.mkdir(parents=True, exist_ok=True)
GEMMA_REVIEW_DIR.mkdir(parents=True, exist_ok=True)

VLLM_PID = WORKING_DIR / "vllm-openai-server.pid"
VLLM_BASE_URL = "http://127.0.0.1:1234/v1"
VLLM_PROCESS = None
QWEN_SERVED_NAME = "vrfai/Qwen3.6-27B-FP8"
GEMMA_SERVED_NAME = "google/gemma-3-27b-it"
QWEN_DATASET_SLUG = "vrfai-qwen3-6-27b-fp8-hf-snapshot"
GEMMA_DATASET_SLUG = "gemma3-llm-cli"
EXPECTED_LOCAL_GAME_COUNT = 25

FORBIDDEN_PRIOR_INPUTS = [
    "historical_transcripts",
    "yesterday_transcripts",
    "routebooks",
    "replays",
    "solved_paths",
    "hidden_labels",
    "cross_run_state",
    "cross_game_state",
]


def _game_key(value):
    """Return the public environment key, never a cross-game lookup key."""
    text = str(value or "").strip()
    match = re.match(r"([A-Za-z0-9]+)", text)
    if not match:
        raise ValueError(f"Cannot derive Duck Harness game key from {value!r}")
    return match.group(1)


def _transcript_game_key(path):
    return _game_key(path.name.split("-", 1)[0])


def _dataset_path(owner, slug):
    try:
        mapped = json.loads(os.environ.get("TAAF_KAGGLE_INPUT_PATHS", "{}"))
    except Exception:
        mapped = {}
    candidates = []
    if f"{owner}/{slug}" in mapped:
        candidates.append(Path(str(mapped[f"{owner}/{slug}"])))
    candidates.extend(
        [
            Path("/kaggle/input/datasets") / owner / slug,
            Path("/kaggle/input") / slug,
        ]
    )
    existing = next((path for path in candidates if path.exists()), None)
    if existing is None:
        raise FileNotFoundError(
            f"Attached dataset {owner}/{slug} was not mounted; checked {candidates}"
        )
    return existing


def _gemma_runtime_paths():
    root = _dataset_path("kehhill", GEMMA_DATASET_SLUG)
    model = root / "gemma3" / "gemma-3-27b-it-q4_0.gguf"
    source_runtime = root / "llamacpp-bin"
    missing = [
        str(path)
        for path in (
            model,
            source_runtime / "llama-server",
            source_runtime / "libllama.so",
            source_runtime / "libggml-cuda.so",
        )
        if not path.exists()
    ]
    if missing:
        raise FileNotFoundError(
            f"Accessible Gemma 3 27B llama.cpp dataset is incomplete: {missing}"
        )

    # Kaggle input datasets are immutable and may strip executable bits. Stage
    # the complete linked runtime in /kaggle/working before launching it.
    if not GEMMA_RUNTIME_DIR.exists():
        shutil.copytree(source_runtime, GEMMA_RUNTIME_DIR)
    server = GEMMA_RUNTIME_DIR / "llama-server"
    server.chmod(server.stat().st_mode | 0o111)
    runtime_missing = [
        str(path)
        for path in (
            server,
            GEMMA_RUNTIME_DIR / "libllama.so",
            GEMMA_RUNTIME_DIR / "libggml-cuda.so",
        )
        if not path.exists()
    ]
    if runtime_missing or not os.access(server, os.X_OK):
        raise RuntimeError(
            "Staged Gemma llama.cpp runtime is not launchable: "
            f"missing={runtime_missing} executable={os.access(server, os.X_OK)}"
        )
    print(
        f"RELAY staged executable Gemma runtime: {server}",
        flush=True,
    )
    return model, server, GEMMA_RUNTIME_DIR


def _server_models():
    with urlopen(VLLM_BASE_URL + "/models", timeout=10) as response:
        payload = json.loads(response.read().decode("utf-8"))
    return [str(item.get("id")) for item in payload.get("data", [])]


def _assert_server(expected):
    model_ids = _server_models()
    if expected not in model_ids:
        raise RuntimeError(f"Expected active model {expected!r}; found {model_ids!r}")
    print(f"RELAY active model verified: {expected}", flush=True)


def _stop_server():
    global VLLM_PROCESS
    if not VLLM_PID.exists():
        VLLM_PROCESS = None
        return
    try:
        pid = int(VLLM_PID.read_text(encoding="utf-8").strip())
    except (OSError, ValueError):
        VLLM_PID.unlink(missing_ok=True)
        return
    if VLLM_PROCESS is not None and VLLM_PROCESS.pid == pid:
        try:
            VLLM_PROCESS.terminate()
            VLLM_PROCESS.wait(timeout=90)
        except subprocess.TimeoutExpired:
            VLLM_PROCESS.kill()
            VLLM_PROCESS.wait(timeout=30)
    else:
        try:
            os.kill(pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        deadline = time.monotonic() + 90
        while time.monotonic() < deadline:
            try:
                state = (Path("/proc") / str(pid) / "stat").read_text().split()[2]
                if state == "Z":
                    break
                os.kill(pid, 0)
            except (FileNotFoundError, ProcessLookupError):
                break
            time.sleep(2)
        else:
            try:
                os.kill(pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
    VLLM_PID.unlink(missing_ok=True)
    VLLM_PROCESS = None
    time.sleep(5)


def _start_server(model_path, served_name, family):
    global VLLM_PROCESS
    log_path = WORKING_DIR / f"relay-server-{family}.log"
    site = WORKING_DIR / "vllm-site-packages"
    env = os.environ.copy()
    if family == "gemma":
        expected_model, server, library_dir = _gemma_runtime_paths()
        if Path(model_path) != expected_model:
            raise RuntimeError(
                f"Gemma model path changed: expected={expected_model} actual={model_path}"
            )
        env["LD_LIBRARY_PATH"] = (
            str(library_dir) + os.pathsep + env.get("LD_LIBRARY_PATH", "")
        )
        command = [
            str(server),
            "--model",
            str(model_path),
            "--alias",
            served_name,
            "--host",
            "127.0.0.1",
            "--port",
            "1234",
            "--ctx-size",
            str(int(os.environ.get("TAAF_CONTEXT_WINDOW", "32768"))),
            "--parallel",
            "1",
            "--jinja",
        ]
    else:
        env["PYTHONPATH"] = str(site) + os.pathsep + env.get("PYTHONPATH", "")
        command = [
            sys.executable,
            "-m",
            "vllm.entrypoints.openai.api_server",
            "--model",
            str(model_path),
            "--served-model-name",
            served_name,
            "--host",
            "127.0.0.1",
            "--port",
            "1234",
            "--tensor-parallel-size",
            "1",
            "--generation-config",
            "vllm",
            "--enable-prefix-caching",
            "--gpu-memory-utilization",
            "0.92",
            "--max-model-len",
            str(int(os.environ.get("TAAF_CONTEXT_WINDOW", "32768"))),
        ]
        command.extend(
            [
                "--enable-auto-tool-choice",
                "--tool-call-parser",
                "qwen3_coder",
                "--reasoning-parser",
                "qwen3",
                "--default-chat-template-kwargs",
                '{"preserve_thinking": true}',
            ]
        )
    handle = log_path.open("w", encoding="utf-8")
    process = subprocess.Popen(
        command,
        env=env,
        stdout=handle,
        stderr=subprocess.STDOUT,
        text=True,
    )
    VLLM_PROCESS = process
    VLLM_PID.write_text(str(process.pid), encoding="utf-8")
    deadline = time.monotonic() + 1200
    while time.monotonic() < deadline:
        if process.poll() is not None:
            tail = log_path.read_text(encoding="utf-8", errors="replace")[-16000:]
            raise RuntimeError(
                f"{family} relay server exited with {process.returncode}\n{tail}"
            )
        try:
            _assert_server(served_name)
            return
        except Exception:
            time.sleep(5)
    tail = log_path.read_text(encoding="utf-8", errors="replace")[-16000:]
    raise TimeoutError(f"{family} relay server did not become ready\n{tail}")


def _patch_qwen_globals():
    import inference.agent.tool_agent as tool_agent_module

    tool_agent_module._LOCAL_ANALYZER_MODEL_ID = QWEN_SERVED_NAME
    tool_agent_module._DEFAULT_ANALYZER_MODEL = QWEN_SERVED_NAME
    tool_agent_module._LOCAL_ANALYZER_BASE_URL = VLLM_BASE_URL
    os.environ["LOCAL_ANALYZER_BASE_URL"] = VLLM_BASE_URL
    os.environ["OPENAI_BASE_URL"] = VLLM_BASE_URL
    os.environ["LOCAL_ANALYZER_MODEL_ID"] = QWEN_SERVED_NAME
    os.environ["INFERENCE_ANALYZER_MODEL"] = QWEN_SERVED_NAME


def _competition_games():
    import arc_agi
    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [info.game_id for info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed no games.")
    if len(set(game_ids)) != len(game_ids):
        raise RuntimeError("Competition Arcade exposed duplicate game IDs.")
    print(
        f"REAL OFFICIAL ARCADE: discovered all {len(game_ids)} gateway games",
        flush=True,
    )
    return [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec)
        for game_id in game_ids
    ]


def _offline_environment_root():
    root = Path(
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files"
    )
    game_dirs = sorted(
        path for path in root.iterdir()
        if path.is_dir() and re.fullmatch(r"[A-Za-z0-9]{4}", path.name)
    )
    if len(game_dirs) != EXPECTED_LOCAL_GAME_COUNT:
        raise RuntimeError(
            f"Competition environment_files must contain {EXPECTED_LOCAL_GAME_COUNT} games; "
            f"found {len(game_dirs)} under {root}"
        )
    print(
        f"REAL LOCAL ARCADE: root={root} games={len(game_dirs)}",
        flush=True,
    )
    return root


def _offline_games():
    import arc_agi
    import taaf.game_api

    environment_root = _offline_environment_root()
    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=str(environment_root),
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=str(environment_root),
    )
    game_ids = [info.game_id for info in arcade.available_environments]
    if len(game_ids) != EXPECTED_LOCAL_GAME_COUNT:
        raise RuntimeError(
            f"Local Arcade must expose {EXPECTED_LOCAL_GAME_COUNT} games; found {len(game_ids)}"
        )
    return [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec)
        for game_id in game_ids
    ]


def _fresh_games():
    return _competition_games() if TRUE_SUBMISSION else _offline_games()


def _wait_for_gateway(base_url, timeout_s=900):
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle competition gateway did not become ready: {last_error}")


def _gemma_chat(prompt):
    payload = {
        "model": GEMMA_SERVED_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.15,
        "max_tokens": 800,
    }
    request = Request(
        VLLM_BASE_URL + "/chat/completions",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urlopen(request, timeout=90) as response:
        message = json.loads(response.read().decode("utf-8"))["choices"][0]["message"]
    content = str(message.get("content") or "").strip()
    if not content:
        raise RuntimeError("Gemma returned an empty same-run handoff.")
    return content


def _review_with_gemma(game_key, transcript):
    # Tail bounding is a context-management operation on this game's current
    # pass-1 transcript; it does not introduce any prior or other-game input.
    transcript_tail = transcript[-60000:]
    prompt = f"""
You are Gemma, the reviewing model in a strict no-prior Duck Harness relay.
This is game {game_key}. You may use ONLY the pass-1 Qwen transcript below,
which was produced moments ago in this same competition run and same game.
Do not import remembered routes, prior submissions, other games, hidden labels,
or external knowledge about this environment.

Review Qwen's observed objects, tested actions, rewards, state changes, failures,
and partial world model. Return a compact handoff to Qwen for a fresh pass-2
Duck Harness run. Include:
- World model
- Goal model
- Action model
- Confirmed findings
- Failed hypotheses or moves to avoid
- Best next-pass plan
Do not emit tool calls and do not claim facts absent from the transcript.

PASS-1 QWEN TRANSCRIPT FOR GAME {game_key}:
{transcript_tail}
""".strip()
    last_error = None
    for attempt in range(1, 3):
        try:
            review = _gemma_chat(prompt)
            print(
                f"RELAY Gemma review complete: game={game_key} "
                f"attempt={attempt} chars={len(review)}",
                flush=True,
            )
            return review
        except Exception as exc:
            last_error = exc
            print(
                f"RELAY Gemma review retry: game={game_key} "
                f"attempt={attempt} error={exc!r}",
                flush=True,
            )
            time.sleep(5 * attempt)
    raise RuntimeError(
        f"Gemma failed to produce a required handoff for {game_key}: {last_error!r}"
    )


def _run_score(run):
    return float(getattr(run, "final_score", None) or 0.0)


def _run_levels(run):
    return int(getattr(run, "levels_completed", 0) or 0)


def _run_actions(run):
    return len(getattr(run, "history", ()) or ())


def _won(run):
    state = getattr(run, "state", "")
    return str(getattr(state, "name", state)).lower().endswith("won")


from inference.agent.tool_agent import ToolAgent

# =====================================================================
# Dual-model mid-game switch: exactly ONE Duck Harness environment pass
# per game. Qwen acts from start to finish. When Qwen stalls (no
# score/level progress within a policy budget), the shared engine swaps
# to Gemma mid-run, Gemma reviews this game's SAME-run transcript (plain
# text, no tools), the engine swaps back to Qwen, and Qwen continues the
# SAME pass seeded with that handoff. No prior or cross-game data is
# used. The final output is the single pass.
# =====================================================================
import threading

from duck_midgame import (
    DualModelSwitchToolAgent,
    MidgameEngineController,
    write_switch_manifest,
)

SWITCH_POLICY = {
    # Minimum executed environment actions before a switch is considered.
    "min_actions_at_all": int(os.environ.get("TAAF_SWITCH_MIN_ACTIONS_AT_ALL", "50")),
    # Executed actions with no score/level progress that trigger a switch.
    "min_actions_before_switch": int(os.environ.get("TAAF_SWITCH_MIN_ACTIONS", "140")),
    # Consecutive analysis steps with no progress that trigger a switch.
    "min_steps_before_switch": int(os.environ.get("TAAF_SWITCH_MIN_STEPS", "16")),
    # Review cycles per game: Qwen -> Gemma review -> Qwen (1 or 2).
    "max_switches": int(os.environ.get("TAAF_SWITCH_MAX_SWITCHES", "1")),
    # Same-run transcript tail (chars) handed to Gemma.
    "handoff_chars": int(os.environ.get("TAAF_SWITCH_HANDOFF_CHARS", "60000")),
    # Assistant turns of raw message history kept after the switch.
    "keep_recent_turns": int(os.environ.get("TAAF_SWITCH_KEEP_RECENT_TURNS", "2")),
    # Tokens budgeted for Gemma's review of the stalled transcript.
    "review_max_tokens": int(os.environ.get("TAAF_SWITCH_REVIEW_MAX_TOKENS", "800")),
    # Minimum remaining per-game budget (seconds) required to attempt a switch.
    "min_remaining_budget_seconds": int(
        os.environ.get("TAAF_SWITCH_MIN_REMAINING_BUDGET", "200")
    ),
}


def _swap_to_gemma():
    """Real engine swap: stop the live vLLM Qwen, start the llama.cpp Gemma."""
    gemma_model_path, _, _ = _gemma_runtime_paths()
    _stop_server()
    _start_server(gemma_model_path, GEMMA_SERVED_NAME, "gemma")


def _swap_to_qwen():
    """Reverse swap (defensive; the default policy is one-way)."""
    qwen_model_path = _dataset_path("driessmit1", QWEN_DATASET_SLUG)
    _stop_server()
    _start_server(qwen_model_path, QWEN_SERVED_NAME, "qwen")


MIDGAME_ENGINE = MidgameEngineController(
    primary={
        "provider": "vllm",
        "base_url": VLLM_BASE_URL,
        "model_id": QWEN_SERVED_NAME,
    },
    secondary={
        "provider": "vllm",
        "base_url": VLLM_BASE_URL,
        "model_id": GEMMA_SERVED_NAME,
    },
    to_secondary=_swap_to_gemma,
    to_primary=_swap_to_qwen,
)


def _midgame_analyzer_factory(game, index):
    key = _game_key(
        getattr(game, "env_name", "")
        or getattr(game, "game_id", "")
        or index
    )
    return DualModelSwitchToolAgent(
        primary={
            "provider": "vllm",
            "base_url": VLLM_BASE_URL,
            "model_id": QWEN_SERVED_NAME,
        },
        secondary={
            "provider": "vllm",
            "base_url": VLLM_BASE_URL,
            "model_id": GEMMA_SERVED_NAME,
        },
        engine=MIDGAME_ENGINE,
        policy=SWITCH_POLICY,
        timeout=bm.solver.analyzer_timeout,
        save_request_logs=bm.solver.save_request_logs,
    )


if TRUE_SUBMISSION:
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    print("REAL OFFICIAL ARCADE: competition gateway ready", flush=True)
else:
    _offline_environment_root()

# Fresh ephemeral output state guarantees that no earlier notebook artifact is
# visible to the run. This only clears this current Kaggle run's working dir.
for stale in (WORKING_DIR / "transcripts").glob("*.txt"):
    stale.unlink()

run_game_apis = _fresh_games()
RUN_GAME_COUNT = len(run_game_apis)
if RUN_GAME_COUNT < 1:
    raise RuntimeError("No games are available for the single-pass switch run.")

# One environment pass per game (vs two in the relay), so each game gets up to
# twice the per-game budget at the same concurrency. Kaggle's nine-hour limit
# still applies; the 1,500-second cap matches the prior scored run's local
# per-game setting whenever the lane count makes it runtime-feasible.
NOTEBOOK_RUNTIME_TARGET_SECONDS = 8 * 60 * 60 + 30 * 60
NON_ENVIRONMENT_RESERVE_SECONDS = 110 * 60
MAX_PER_GAME_PASS_SECONDS = float(
    os.environ.get("TAAF_MAX_PER_GAME_SECONDS", "3000")
)
available_environment_seconds = (
    NOTEBOOK_RUNTIME_TARGET_SECONDS - NON_ENVIRONMENT_RESERVE_SECONDS
)
runtime_safe_per_game = max(
    45.0,
    available_environment_seconds * TARGET_CONCURRENCY / float(RUN_GAME_COUNT),
)
RUN_PER_GAME_SECONDS = min(
    MAX_PER_GAME_PASS_SECONDS,
    runtime_safe_per_game,
    _original_game_budget if _original_game_budget > 0 else runtime_safe_per_game,
)
bm.solver.max_runtime_s_per_game = RUN_PER_GAME_SECONDS
bm.games = run_game_apis
bm.n_passes = 1
bm.game_weights = None
bm.solver.concurrency = TARGET_CONCURRENCY

print(
    "REAL RUNTIME BUDGET: "
    f"games={RUN_GAME_COUNT} passes=1 concurrency={TARGET_CONCURRENCY} "
    f"per_game_seconds={RUN_PER_GAME_SECONDS:.3f} "
    f"environment_upper_bound_seconds="
    f"{RUN_GAME_COUNT * RUN_PER_GAME_SECONDS / TARGET_CONCURRENCY:.3f}",
    flush=True,
)

# Keep every created analyzer for the switch manifest, then hand the factory
# to the solver (the factory survives the solver's deepcopy).
agents_by_key: dict[str, DualModelSwitchToolAgent] = {}


def _collect_analyzer_factory(game, index):
    key = _game_key(
        getattr(game, "env_name", "")
        or getattr(game, "game_id", "")
        or index
    )
    agent = _midgame_analyzer_factory(game, index)
    agents_by_key[key] = agent
    return agent


bm.solver.analyzer_factory = _collect_analyzer_factory
switch_manifest_path = WORKING_DIR / "midgame_switch_manifest.json"

try:
    # Fail fast on both model runtimes before spending hours on the pass. Qwen
    # is already live from setup; briefly smoke-test the accessible Gemma 3 27B
    # GGUF/llama.cpp runtime, then restore Qwen for the environment pass.
    gemma_model_path, _, _ = _gemma_runtime_paths()
    _stop_server()
    _start_server(gemma_model_path, GEMMA_SERVED_NAME, "gemma")
    gemma_smoke = _gemma_chat(
        "Reply with exactly GEMMA_RELAY_READY and no other text."
    )
    if "GEMMA_RELAY_READY" not in gemma_smoke:
        raise RuntimeError(f"Gemma relay smoke test failed: {gemma_smoke!r}")
    print(
        f"MIDGAME Gemma smoke test real model output: {gemma_smoke}",
        flush=True,
    )
    _stop_server()
    qwen_model_path = _dataset_path("driessmit1", QWEN_DATASET_SLUG)
    _start_server(qwen_model_path, QWEN_SERVED_NAME, "qwen")
    _assert_server(QWEN_SERVED_NAME)
    _patch_qwen_globals()
    print(
        "MIDGAME PASS 1/1: Qwen acts every game; Gemma reviews mid-run on stall; "
        "engine returns to Qwen after each review",
        flush=True,
    )
    await bm.run(
        soft_end_time=None,
        runtime_environment=target,
        minimal_diagnostics=True,
    )

    runs = {_game_key(run.game_id): run for run in bm.game_runs}
    if len(runs) != RUN_GAME_COUNT:
        raise RuntimeError(
            f"Single pass must complete all {RUN_GAME_COUNT} games; found {len(runs)}"
        )
    switched_keys = sorted(
        key for key, agent in agents_by_key.items() if agent.is_switched
    )
    for key in sorted(runs):
        run = runs[key]
        agent = agents_by_key.get(key)
        model_flag = "qwen+gemma" if (agent is not None and agent.is_switched) else "qwen-only"
        print(
            "OFFICIAL SCORE "
            f"game={key} pass=1 models={model_flag} score={_run_score(run):.6f} "
            f"levels={_run_levels(run)} actions={_run_actions(run)}",
            flush=True,
        )
    print(
        "OFFICIAL AGGREGATE "
        f"games={RUN_GAME_COUNT} environment_executions={RUN_GAME_COUNT} "
        f"switched_games={len(switched_keys)} "
        f"score_sum={sum(_run_score(run) for run in runs.values()):.6f}",
        flush=True,
    )

    write_switch_manifest(list(agents_by_key.values()), switch_manifest_path)
    print(
        f"MIDGAME manifest written: {switch_manifest_path} switched={switched_keys}",
        flush=True,
    )

    # Keep one selected result per public game key in the required root artifact.
    import pandas as pd

    rows = [
        {
            "row_id": f"{run.game_id}_0",
            "game_id": str(run.game_id),
            "end_of_game": _won(run),
            "score": _run_score(run),
        }
        for run in runs.values()
    ]
    if len(rows) != RUN_GAME_COUNT:
        raise RuntimeError(
            f"Official artifact must contain {RUN_GAME_COUNT} rows; found {len(rows)}"
        )
    submission_path = WORKING_DIR / "submission.parquet"
    pd.DataFrame(
        rows,
        columns=["row_id", "game_id", "end_of_game", "score"],
    ).to_parquet(submission_path, index=False)
    written = pd.read_parquet(submission_path)
    required_columns = ["row_id", "game_id", "end_of_game", "score"]
    if list(written.columns) != required_columns:
        raise RuntimeError(
            f"Submission columns are invalid: {list(written.columns)}"
        )
    if len(written) != RUN_GAME_COUNT:
        raise RuntimeError(
            f"Submission must contain {RUN_GAME_COUNT} real game rows; "
            f"found {len(written)}"
        )
    if written["game_id"].astype(str).nunique() != RUN_GAME_COUNT:
        raise RuntimeError("Submission contains duplicate real game IDs")
    if written["score"].isna().any():
        raise RuntimeError("Submission contains missing scores")
    print(
        "REAL SCORE ARTIFACT VALIDATED: "
        f"path={submission_path} rows={len(written)} "
        f"score_sum={float(written['score'].sum()):.6f}",
        flush=True,
    )
    print(
        "MIDGAME COMPLETE: exactly one Duck Harness pass per game; "
        f"mid-game switch used on {len(switched_keys)} games",
        flush=True,
    )
finally:
    _stop_server()
    for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
        print("Running teardown:", command, flush=True)
        subprocess.run(
            command,
            shell=True,
            check=False,
            cwd=WORKING_DIR,
            env=_command_env(),
        )


## 8. Show the diagnostics

A non-submission run writes `diagnostics.html` to `/kaggle/working`; it is rendered inline below
(and downloadable from the working directory). You should be able to click around through the links.

In [10]:
from html import escape

from IPython.display import HTML, display

diagnostics_html = WORKING_DIR / "diagnostics.html"
if diagnostics_html.is_file():
    # Isolate the full document in an iframe so its styles don't leak into the notebook.
    display(
        HTML(
            f'<iframe srcdoc="{escape(diagnostics_html.read_text(), quote=True)}" '
            'width="100%" height="900" style="border:0"></iframe>'
        )
    )
else:
    print("No diagnostics.html — minimal diagnostics (real submission) suppresses it.")


In [ ]:
# === CLEAR LOCAL SCORE CARD ===
# Keep this as the final cell. It summarizes the completed local/offline run directly from `bm`.
import re
from html import escape

from IPython.display import HTML, display

if TRUE_SUBMISSION:
    print("Official competition rerun: the local estimate card is intentionally hidden.")
elif not getattr(bm, "game_runs", None):
    print("No completed local benchmark data is available yet.")
else:
    from taaf import diagnostics as taaf_diagnostics

    _summary = taaf_diagnostics.run_summary_text(bm)

    def _extract(pattern, cast=str, default=None):
        match = re.search(pattern, _summary, flags=re.MULTILINE)
        if not match:
            return default
        try:
            return cast(match.group(1).strip())
        except Exception:
            return default

    _mean = _extract(r"^mean score:\s*([0-9.]+)\s*$", float, 0.0)
    _median = _extract(r"^median score:\s*([0-9.]+)\s*$", float, 0.0)
    _duration = _extract(r"^duration:\s*(.+?)\s*$", str, "unknown")
    _games = _extract(r"^games:\s*(\d+)\s*$", int, 0)
    _won = _extract(r"^runs:\s*\d+\s*\(won:\s*(\d+)\)\s*$", int, 0)
    _actions = _extract(r"^total actions:\s*(\d+)\s*$", int, 0)
    _tokens = _extract(r"^total tokens:\s*(\d+)\s*$", int, 0)

    _per_game = re.findall(
        r"^\s+\S+:\s+score=([0-9.]+),\s+levels=([0-9.]+)/([0-9.]+),",
        _summary,
        flags=re.MULTILINE,
    )
    _positive = sum(float(score) > 0 for score, _, _ in _per_game)
    _levels_done = sum(float(done) for _, done, _ in _per_game)
    _levels_total = sum(float(total) for _, _, total in _per_game)

    _budget_s = float(globals().get("RUN_PER_GAME_SECONDS", 0.0) or 0.0)
    _budget_label = (
        f"{_budget_s / 60:.1f} min/game"
        if _budget_s > 0
        else "full local budget"
    )
    _level_label = (
        f"{_levels_done:.0f}/{_levels_total:.0f}"
        if _levels_total > 0
        else "unknown"
    )
    _positive_label = (
        f"{_positive}/{len(_per_game)}"
        if _per_game
        else "unknown"
    )

    display(HTML(f"""
    <div style="border:1px solid #6b7280;border-radius:14px;padding:20px 24px;margin:14px 0;max-width:920px;font-family:Arial,sans-serif">
      <div style="font-size:15px;font-weight:800;letter-spacing:.05em">ARC-AGI-3 MID-GAME SWITCH LOCAL SCORE</div>
      <div style="font-size:46px;font-weight:850;line-height:1.15;margin-top:8px">{_mean:.2f}<span style="font-size:18px;font-weight:500"> / 100</span></div>
      <div style="font-size:14px;margin-top:4px">Estimated mean score on the local public environments (not the official hidden-environment leaderboard score)</div>
      <hr style="margin:16px 0;border:none;border-top:1px solid #6b7280">
      <table style="border-collapse:collapse;width:100%;font-size:14px;line-height:1.9">
        <tr><td>Median score</td><td><b>{_median:.2f}</b></td><td>Games fully solved</td><td><b>{_won}/{_games}</b></td></tr>
        <tr><td>Games with positive score</td><td><b>{_positive_label}</b></td><td>Levels completed</td><td><b>{_level_label}</b></td></tr>
        <tr><td>Total actions</td><td><b>{_actions:,}</b></td><td>Total generated tokens</td><td><b>{_tokens:,}</b></td></tr>
        <tr><td>Benchmark duration</td><td><b>{escape(_duration)}</b></td><td>Per-pass game budget</td><td><b>{escape(_budget_label)}</b></td></tr>
      </table>
      <div style="font-size:12px;margin-top:14px;opacity:.78">Every discovered local game completed one Duck Harness environment pass. Qwen started every game; Gemma took over mid-run only on stalls, seeded with a same-run transcript handoff, and the displayed score is that single pass.</div>
    </div>
    """))
